# Preprocessing Student Answer
Create scoring web apps and validate the scoring result.

In [23]:
# Load environment variables from .env file
import os
from dotenv import load_dotenv

# Load .env file from parent directory
load_dotenv("../.env")

print("✓ Environment variables loaded")

✓ Environment variables loaded


In [24]:
from google import genai
from google.genai import types

# Get API key from environment variable
API_KEY = os.getenv("GOOGLE_GENAI_API_KEY")

if not API_KEY or API_KEY == "your-api-key-here":
    raise ValueError(
        "Please set your GOOGLE_GENAI_API_KEY in the .env file. "
        "Get your API key from: https://aistudio.google.com/apikey"
    )

# Initialize client with Vertex AI Express Mode
client = genai.Client(vertexai=True, api_key=API_KEY)

print("✓ Vertex AI Express Mode initialized successfully!")

✓ Vertex AI Express Mode initialized successfully!


## Setup Vertex AI Express Mode with API Key

This notebook now uses **Vertex AI Express Mode** with API key authentication instead of OAuth/ADC.

**Steps to get your API key:**
1. Visit https://aistudio.google.com/apikey
2. Create or select your API key
3. Copy the API key and add it to the `.env` file in the parent directory:
   ```
   GOOGLE_GENAI_API_KEY=your-actual-api-key-here
   ```

**Benefits of Express Mode:**
- ✓ Simpler authentication (just an API key)
- ✓ No need for gcloud CLI authentication
- ✓ No service account JSON files
- ✓ Easy to use in notebooks and scripts

In [25]:
prefix = "VTC Test"
pdf_file = "../data/demo.pdf"

# Load marking scheme Excel file generated by get_marking_scheme_excel.ipynb
marking_scheme_file = f"../sample/{prefix} Marking Scheme.xlsx"
standard_answer = marking_scheme_file

In [26]:
import os

file_name = os.path.basename(pdf_file)
file_name = os.path.splitext(file_name)[0]
base_path = "../marking_form/" + file_name
base_path_images = base_path + "/images/"
base_path_annotations = base_path+"/annotations/"
base_path_questions = base_path+"/questions"
base_path_javascript = base_path+"/javascript"

# create directory tree for base_path_images
os.makedirs(base_path_images, exist_ok=True)
os.makedirs(base_path_annotations, exist_ok=True)
os.makedirs(base_path_questions, exist_ok=True)
os.makedirs(base_path_javascript, exist_ok=True)

In [27]:
import json
annotations_path = base_path_annotations + "annotations.json"
with open(annotations_path, "r") as f: 
    annotations = json.load(f)          

#flatten annotations to list 
annotations_list = []
for page in annotations:
    for annotation in annotations[page]:
        annotation["page"] = int(page)
        # x to left, y to top
        annotation["left"] = annotation["x"]
        annotation["top"] = annotation["y"]
        annotation.pop("x")
        annotation.pop("y")
        annotations_list.append(annotation) 
annotations_list

# convert annotations_list to dict with key with label
annotations_dict = {}
for annotation in annotations_list:
    annotations_dict[annotation["label"]] = annotation
# annotations_dict


In [28]:
# extract list of label from annotations as questions
questions = []
for annotation in annotations_list:
    if annotation["label"] not in questions:
        questions.append(annotation["label"])
# remove 'NAME', 'ID', 'CLASS' if exists in questions
if 'NAME' in questions:
    questions.remove('NAME')
if 'ID' in questions:
    questions.remove('ID')
if 'CLASS' in questions:
    questions.remove('CLASS')    

# sort questions 
questions.sort()
question_with_answer = questions.copy()
questions = ['NAME', 'ID', 'CLASS'] + questions
questions

['NAME', 'ID', 'CLASS', 'Q1', 'Q2', 'Q3', 'Q4', 'Q5']

## Validate Provided Standard Answer for each question

In [29]:
## Load data directly from Marking Scheme Excel
import pandas as pd

# Define name list file path
name_list_file = f"../sample/{prefix} Name List.xlsx"

# Try to load Name List sheet from name list file
try:
    name_list_df = pd.read_excel(name_list_file, sheet_name="Name List")
    print(f"✓ Loaded Name List from: {name_list_file}")
except Exception as e:
    print(f"⚠️ Failed to load Name List: {e}")
    name_list_df = None

# Load Marking Scheme sheet - this contains all the data needed
marking_scheme_df = pd.read_excel(standard_answer, sheet_name="Marking Scheme")
print(f"✓ Loaded Marking Scheme directly")
print(f"  Columns: {list(marking_scheme_df.columns)}")

# Create Answer sheet dictionary for backward compatibility
# Map question_number to marking_scheme for standard_answer lookup
standard_answer_df = marking_scheme_df[['question_number', 'marking_scheme', 'marks']].copy()
standard_answer_df.columns = ['Question', 'Answer', 'Mark']

print(f"✓ Prepared data for scoring")
standard_answer_df.head()

✓ Loaded Name List from: ../sample/VTC Test Name List.xlsx
✓ Loaded Marking Scheme directly
  Columns: ['question_number', 'question_text', 'marking_scheme', 'marks']
✓ Prepared data for scoring


,Question,Answer,Mark
0,Q1,"[2 marks] Correctly stating ""Vocational and Pr...",10
1,Q2,[5 marks] Correctly identifying that IVE offer...,10
2,Q3,"[3 marks] Explaining ""Think"" (Theory/Academic ...",10
3,Q4,"[5 marks] Correctly naming the ""Diploma of Fou...",10
4,Q5,[4 marks] General explanation (Curriculum rele...,10


Covert Question to str

In [30]:
standard_answer_df["Question"] = standard_answer_df["Question"].astype(str)

In [31]:
from termcolor import colored

# check question_with_answer in standard_answer_df Question column
for question in question_with_answer:
    if question not in standard_answer_df["Question"].values:
        print(colored("Question {} is not in standard_answer!".format(question), 'red'))

for question in standard_answer_df["Question"].values:
    if question not in question_with_answer:
        print(colored("Question {} is not in annotations!".format(question), 'red'))
            

In [32]:
standard_answer = standard_answer_df.set_index("Question").to_dict()["Answer"]
standard_answer

{'Q1': '[2 marks] Correctly stating "Vocational and Professional Education and Training". [4 marks] Explaining that it focuses on practical skills or specialized trades. [4 marks] Explaining the benefit to the workforce (reducing skills gap, employment readiness). General Grading: 9-10 marks for complete/accurate answers; 6-8 marks for mostly correct but missing details; 3-5 marks for basic understanding; 0-2 marks for incorrect/irrelevant answers.\n\n--- General Grading Guide ---\n9-10 marks: The answer is complete, accurate, uses correct terminology, and is well-explained. 6-8 marks: The answer is mostly correct but misses a specific detail (e.g., forgets the full name of a diploma) or the explanation is slightly vague. 3-5 marks: The student shows basic understanding but misses the core point or only answers half the q. 0-2 marks: The answer is largely incorrect, irrelevant, or blank.',
 'Q2': "[5 marks] Correctly identifying that IVE offers Higher Diplomas/Technical training. [5 ma

In [33]:
standard_mark = standard_answer_df.set_index("Question").to_dict()["Mark"]
standard_mark

{'Q1': 10, 'Q2': 10, 'Q3': 10, 'Q4': 10, 'Q5': 10}

Check for the regeneration of question.

In [34]:
import os
import json

questionAndControl = {}
for path, currentDirectory, files in os.walk(base_path_questions):
    for file in files:
        if file == "control.json":
            question = path[len(base_path_questions) + 1 :]
            f = open(os.path.join(path, file))
            data = json.load(f)
            if "regenerate" in data:
                questionAndControl[question] = data
            f.close()

questionAndControl

{}

In [35]:
from distutils.dir_util import copy_tree
import shutil
import os

from_directory = os.path.join(os.getcwd(), "..","templates", "javascript")
copy_tree(from_directory, base_path_javascript)
ico = os.path.join(os.getcwd(), "..","templates", "favicon.ico")
# copy ico file  to base_path
shutil.copyfile(ico, base_path+"/favicon.ico")

'../marking_form/demo/favicon.ico'

Generate the index.html

In [36]:
from pathlib import Path
from jinja2 import Environment, FileSystemLoader

file_loader = FileSystemLoader("../templates")
env = Environment(loader=file_loader)
template = env.get_template("index.html")

output = template.render(
    studentsScriptFileName=file_name,
    textAnswer=questions,
    optionAnswer=[],
)
# open text file
path = Path(os.path.join(base_path, "index.html"))
text_file = open(path, "w")
text_file.write(output)
text_file.close()

In [ ]:
def ocr(prompt: str, filePath: str):
    """
    OCR function using Vertex AI Express Mode
    
    Args:
        prompt: The prompt describing what to extract
        filePath: Path to the image file
    
    Returns:
        Extracted text as string
    """
    # Read the image file
    with open(filePath, "rb") as f:
        data = f.read()
    
    # Create configuration
    config = types.GenerateContentConfig(
        temperature=0,
        top_p=0.5,
        max_output_tokens=2048*2,
        safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HATE_SPEECH",
                threshold="BLOCK_ONLY_HIGH",
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_DANGEROUS_CONTENT",
                threshold="BLOCK_ONLY_HIGH",
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                threshold="BLOCK_ONLY_HIGH",
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_HARASSMENT",
                threshold="BLOCK_ONLY_HIGH",
            ),
        ]
    )
    
    # Generate content
    response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=[
            {
                "role": "user",
                "parts": [
                    {"inline_data": {"mime_type": "image/png", "data": data}},
                    {"text": prompt}
                ]
            }
        ],
        config=config,
    )
    
    return response.text if response.text else ""

In [38]:
import tempfile
from PIL import Image, ImageEnhance

def ocr_image_from_file(question, image_path, left, top, width, height):
    if question == "NAME" :
        return ""
    
    imageFile = tempfile.NamedTemporaryFile(suffix=".png").name
    with Image.open(image_path) as im:
        # The crop method from the Image module takes four coordinates as input.
        # The right can also be represented as (left+width)
        # and lower can be represented as (upper+height).
        (left, top, right, lower) = (
            left,
            top,
            left + width,
            top + height,
        )
        # Here the image "im" is cropped and assigned to new variable im_crop
        im_crop = im.crop((left, top, right, lower))
        imageEnhance = ImageEnhance.Sharpness(im_crop)
        # showing resultant image
        im_crop = imageEnhance.enhance(3)
        im_crop.save(imageFile, format="png")
        
    if question == "ID" :
        text_message = """
            Extract text in this image.
            It is a Student ID in 9 digit number.
            Answer just extracted Student ID and don't answer anything else.
            If you cannot extract Student ID, please return 'No text found!!!'.
            """
    else:    
        text_message ="""
            Extract text in this image in English and number.
            Answer just extracted text and don't answer anything else.
            If you cannot extract text, please return 'No text found!!!'."""       

    try:
        ocr_text = ocr(text_message, imageFile)  
        print(question, image_path ,ocr_text) 
        if ocr_text == "No text found!!!":
            return ""
        return ocr_text
    except Exception as e:
        print(question, image_path, e)    
        return ""

In [53]:
from pydantic import BaseModel, Field

class SimilarityScoreResponse(BaseModel):
    """Structured response for similarity scoring"""
    score: float = Field(description="Similarity score from 0 to 1")
    reasoning: str = Field(description="Brief explanation of the score")

def similarity_score(standard_answer_text, submitted_answer, marking_scheme_text="", total_marks=0):
    """
    Calculate similarity score using Vertex AI Express Mode with structured output
    
    Args:
        standard_answer_text: The correct answer
        submitted_answer: The student's answer
        marking_scheme_text: Detailed marking scheme/rubric (optional)
        total_marks: Total marks available (optional)
    
    Returns:
        Tuple of (score, reasoning)
    """
    # Build prompt with marking scheme context if available
    if marking_scheme_text:
        prompt = f'''You are an expert grader. Evaluate the student's answer based on the marking scheme and rubric provided.

MARKING SCHEME & RUBRIC:
{marking_scheme_text}

Total Marks: {total_marks}

STANDARD/CORRECT ANSWER:
{standard_answer_text}

STUDENT'S SUBMITTED ANSWER:
{submitted_answer}

Evaluate how well the student's answer matches the standard answer according to the marking scheme.
Consider partial credit possibilities outlined in the marking scheme.
Provide a score from 0 to 1 and brief reasoning.'''
    else:
        prompt = f'''You are a grader. Evaluate the similarity between the standard answer and submitted answer.
Provide a score from 0 to 1 (where 0 means completely different and 1 means identical/correct).

Standard answer:
{standard_answer_text}

Submitted Answer:
{submitted_answer}

Provide a score and brief reasoning.'''
    
    config = types.GenerateContentConfig(
        temperature=0,
        top_p=0.3,
        max_output_tokens=1024*8,
        response_mime_type="application/json",
        response_schema=SimilarityScoreResponse,
        safety_settings=[
            types.SafetySetting(
                category="HARM_CATEGORY_HATE_SPEECH",
                threshold="BLOCK_ONLY_HIGH",
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_DANGEROUS_CONTENT",
                threshold="BLOCK_ONLY_HIGH",
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_SEXUALLY_EXPLICIT",
                threshold="BLOCK_ONLY_HIGH",
            ),
            types.SafetySetting(
                category="HARM_CATEGORY_HARASSMENT",
                threshold="BLOCK_ONLY_HIGH",
            ),
        ]
    )
    
    retry = 0
    while retry < 3:
        try:
            response = client.models.generate_content(
                model="gemini-3-flash-preview",
                contents=[{"role": "user", "parts": [{"text": prompt}]}],
                config=config,
            )
            
            # Extract score and reasoning from structured response
            if hasattr(response, 'parsed') and response.parsed is not None:
                score = response.parsed.score
                reasoning = response.parsed.reasoning
                # Ensure score is between 0 and 1
                score = max(0.0, min(1.0, score))
                return score, reasoning
            else:
                # Fallback to text parsing if structured output not available
                text = response.text if response.text else ""
                try:
                    import json
                    parsed = json.loads(text)
                    score = float(parsed.get('score', 0))
                    reasoning = parsed.get('reasoning', 'N/A')
                    score = max(0.0, min(1.0, score))
                    return score, reasoning
                except:
                    print("Retry")
                    retry += 1
                    continue
        except Exception as e:
            print(f"Error in similarity_score: {e}")
            retry += 1
            continue
    
    return 0, "Error: Could not retrieve scoring"


def calculate_similarity(answers, question):
    # Add the standard answer to the head of list.
    if question not in standard_answer:
        ## return list of (0, "") in len of answers
        return [(0, "")] * len(answers)
    
    answer = standard_answer[question]
    
    # Get marking scheme context if available
    marking_scheme_text = ""
    total_marks = 0
    if marking_scheme_df is not None:
        try:
            scheme_row = marking_scheme_df[marking_scheme_df['question_number'].astype(str) == str(question)]
            if not scheme_row.empty:
                marking_scheme_text = scheme_row.iloc[0]['marking_scheme']
                total_marks = scheme_row.iloc[0]['marks']
        except:
            pass
    
    scores_and_reasoning = []
    for submitted_answer in answers:
        submitted_answer = str(submitted_answer)
        if submitted_answer.strip() == "":
            scores_and_reasoning.append((0, "Empty answer"))
            continue
        score, reasoning = similarity_score(answer, submitted_answer, marking_scheme_text, total_marks)
        scores_and_reasoning.append((score, reasoning))
    
    return scores_and_reasoning

In [54]:
import os
import pandas as pd


def get_the_list_of_files(path):
    """
    Get the list of files in the directory
    """
    files = []
    for dirpath, dirnames, filenames in os.walk(path):
        files.extend(filenames)
        break
    return sorted(files)


images = get_the_list_of_files(base_path_images)

# get max page from annotations_list
max_page = 0
for annotation in annotations_list:
    if annotation["page"] > max_page:
        max_page = annotation["page"]
max_page = max_page + (1 if max_page % 2 == 1 else max_page + 2) # Scanner will have a blank page!

# filter images by file name divided by page
images_by_page = []
for page in range(max_page):
    images_by_page.append([])
    for image in images:
        p = int(image.split(".")[0])
        if p % max_page == page:
            images_by_page[page].append(image)


def get_df(question):
    row = annotations_dict[question].copy()
    row["Similarity"] = 0
    row["Reasoning"] = ""
    row["Image"] = images_by_page[row["page"]]
    # append base_path_images to each image
    row["Image"] = ["images/" + image for image in row["Image"]]

    # expend row to dataframe for each image in row["Image"]
    data = pd.DataFrame(row)
    data = data.explode("Image")
    data = data.reset_index(drop=True)

    data["Answer"] = data.apply(
        lambda row: ocr_image_from_file(question,
            base_path + "/" + row["Image"],
            row["left"],
            row["top"],
            row["width"],
            row["height"],
        ),
        axis=1,
    )
    # add column RowNumber
    data["RowNumber"] = data.index + 1
    data["maskPage"] = data["page"]

    scores_and_reasoning = calculate_similarity(data["Answer"].tolist(), question)
    
    # Unpack scores and reasoning
    data["Similarity"] = [item[0] for item in scores_and_reasoning]
    data["Reasoning"] = [item[1] for item in scores_and_reasoning]

    data["page"] = data["Image"].apply(
        lambda x: x.replace("images/", "").replace(".jpg", "")
    )
    data["Mark"] = data["Answer"].apply(lambda x: "0" if len(x.strip()) == 0 else "")

    return data


def save_template_output(output, question, filename):
    path = Path(base_path_questions, question)
    path.mkdir(parents=True, exist_ok=True)
    path = Path(os.path.join(path, filename))
    text_file = open(path, "w")
    text_file.write(output)
    text_file.close()


# question = "NAME"
# get_df(question)

Generate individual question page.

In [56]:
from ipywidgets import IntProgress
from IPython.display import display

max_count = len(questions)
f = IntProgress(min=0, max=max_count) # instantiate the bar
display(f) # display the bar

for question in questions:
    dataTable = get_df(question)
    os.makedirs(base_path_questions + "/" + question, exist_ok=True)
    dataTable.to_csv(base_path_questions + "/" + question + "/data.csv", index=False)

    if question == "ID" or question == "NAME" or question == "CLASS":
        template = env.get_template("questions/index-answer.html")
    else:
        template = env.get_template("questions/index.html")
    output = template.render(
        studentsScriptFileName=file_name,
        question=question,
        standardAnswer=standard_answer[question] if question in standard_answer else "",
        standardMark=standard_mark[question] if question in standard_mark else "",
        estimatedBoundingBox=annotations_dict[question],
        dataTable=dataTable,
    )
    save_template_output(output, question, "index.html")

    template = env.get_template("questions/question.js")
    output = template.render(
        dataTable=dataTable,
        estimatedBoundingBox=annotations_dict[question],
    )
    save_template_output(output, question, "question.js")

    template = env.get_template("questions/style.css")
    output = template.render(
        dataTable=dataTable,
    )
    save_template_output(output, question, "style.css")
    f.value += 1

IntProgress(value=0, max=8)

ID ../marking_form/demo/images/0.jpg 123456789
ID ../marking_form/demo/images/2.jpg 987654321
ID ../marking_form/demo/images/4.jpg 234567890
ID ../marking_form/demo/images/6.jpg 345678912
CLASS ../marking_form/demo/images/0.jpg A
CLASS ../marking_form/demo/images/2.jpg B
CLASS ../marking_form/demo/images/4.jpg C
CLASS ../marking_form/demo/images/6.jpg D
Q1 ../marking_form/demo/images/0.jpg Q1
Vocational and Professional
Education and Traing
Q1 ../marking_form/demo/images/2.jpg Q1
Vacational and professional
Eduate Trarning
Q1 ../marking_form/demo/images/4.jpg 1. Q1
2. Hong Kong skilled labor force
Q1 ../marking_form/demo/images/6.jpg 1. Q1
2. Vocational and Professional
3. Education and Training
Q2 ../marking_form/demo/images/0.jpg Q2
IVE is Highed Diploma
THEi is Degree
Q2 ../marking_form/demo/images/2.jpg Q2
HD is IVE
Degree is THEi
Q2 ../marking_form/demo/images/4.jpg 1. Q2
2. IVE is VTC
3. thei is also VTC
4. Q3
Q2 ../marking_form/demo/images/6.jpg Q2
higher Diploma for IVE
Degree 

In [48]:
from ipywidgets import IntProgress
from IPython.display import display
import pandas as pd

max_count = len(questions)
f = IntProgress(min=0, max=max_count) # instantiate the bar
display(f) # display the bar

for question in questions:
    data_path = base_path_questions + "/" + question + "/data.csv"
    dataTable = pd.read_csv(data_path)
    dataTable = dataTable.replace(".*No text found!!!.*", "", regex=True)
    
    scores_and_reasoning = calculate_similarity(dataTable["Answer"].tolist(), question)
    dataTable["Similarity"] = [item[0] for item in scores_and_reasoning]
    dataTable["Reasoning"] = [item[1] for item in scores_and_reasoning]
    
    dataTable.to_csv(base_path_questions + "/" + question + "/data.csv", index=False)

    if question == "ID" or question == "NAME" or question == "CLASS":
        template = env.get_template("questions/index-answer.html")
    else:
        template = env.get_template("questions/index.html")
    output = template.render(
        studentsScriptFileName=file_name,
        question=question,
        standardAnswer=standard_answer[question] if question in standard_answer else "",
        standardMark=standard_mark[question] if question in standard_mark else "",
        estimatedBoundingBox=annotations_dict[question],
        dataTable=dataTable,
    )
    save_template_output(output, question, "index.html")

    template = env.get_template("questions/question.js")
    output = template.render(
        dataTable=dataTable,
        estimatedBoundingBox=annotations_dict[question],
    )
    save_template_output(output, question, "question.js")

    template = env.get_template("questions/style.css")
    output = template.render(
        dataTable=dataTable,
    )
    save_template_output(output, question, "style.css")
    f.value += 1

IntProgress(value=0, max=8)

## Validate Student ID

In [57]:
# load csv file to dataframe
import pandas as pd

id_from_oscr = pd.read_csv(base_path_questions + "/" + "ID" + "/data.csv")["Answer"].tolist()
id_from_oscr = [str(int(float(x))) if pd.notna(x) else x for x in id_from_oscr]

id_from_namelist = name_list_df["Student"].to_list()

# check duplicate id
duplicate_id = []
for id in id_from_oscr:
    if id_from_oscr.count(id) > 1:
        duplicate_id.append(id)
duplicate_id = list(set(duplicate_id))
if len(duplicate_id) > 0:
    print(colored("Duplicate ID: {}".format(duplicate_id), "red"))

id_from_oscr = [str(id) for id in id_from_oscr]
id_from_namelist = [str(id) for id in id_from_namelist]

# compare oscr_id and validate_id
ocr_missing_id = []
name_list_missing_id = []
for id in id_from_oscr:    
    if id not in id_from_namelist:       
        name_list_missing_id.append(id)

for id in id_from_namelist:
    if id not in id_from_oscr:   
        ocr_missing_id.append(id)

## OCR scan error case

In [58]:
from termcolor import colored
if len(ocr_missing_id) > 0:
    print(colored("Some IDs OCR is not in NameList and you need to fix it manually!", "red"))
    for id in name_list_missing_id:
        print(colored(id, "red"))

## Potential Absent Case

In [59]:
from termcolor import colored

if len(ocr_missing_id) > 0:
    print(colored("Number of absentee {}.".format(len(ocr_missing_id)), "red"))
    print(colored("ID in Name List does not find from OCR!", "red"))
    for id in ocr_missing_id:
        print(colored(id, "red"))

# Start Python HTTPServer

The webserver log is in output/server.log.

If you are in development and don't want the notebook being blocked by running webserver, you can open a terminal and run the below command.

file_name=XXXX python server.py 8000

In [52]:
print("file_name={} python server.py".format(file_name))

file_name=demo python server.py


In [ ]:
# You can also uncomment the following line to run the web server but if it crashes, you need to restart the kernel.
# !cd .. && file_name=TestScript python server.py

# Post Processing after scoring
1. Check all question has scores.
2. Check ID again.
3. Remove version history.

In [60]:
# check each sub folder of base_path_questions contains file name mark.json, ignore the root folder
import os
import json

unfinsihed_scoring = []
for path, currentDirectory, files in os.walk(base_path_questions):
    if path != base_path_questions:
        # extract question name from path
        question = path[len(base_path_questions) + 1 :]
        if "mark.json" not in files:
            unfinsihed_scoring.append(question)
        else:
            # read mark.json as json
            with open(os.path.join(path, "mark.json"), "r") as f:
                marks = json.load(f)            
            # check each mark in marks that attribute "mark" or "overridedMark" is not empty
            for mark in marks:
                if mark['mark'] == "" and  mark['overridedMark'] == "":
                    # extract question name from path                   
                    unfinsihed_scoring.append(question)
                    break             

if len(unfinsihed_scoring) > 0:            
    print(colored("{} have some question not yet mark!".format(unfinsihed_scoring), "red"))          
else:
    print("All questions have been marked!")

All questions have been marked!


Check ID

In [62]:
import os
import json
from termcolor import colored

with open(os.path.join(base_path_questions,"ID", "mark.json"), "r") as f:
    marks = json.load(f)

id_from_mark = list(map(lambda x: x["overridedMark"] if x["overridedMark"] != "" else x["mark"], marks))
id_from_namelist = name_list_df["Student"].to_list()

# convert id_from_mark to string
id_from_mark = [str(id) for id in id_from_mark]
id_from_namelist = [str(id) for id in id_from_namelist]

mark_missing_id = []
for id in id_from_namelist:
    if id not in id_from_mark:   
        mark_missing_id.append(id)
print(colored("In class but not marked - {}!".format(mark_missing_id), "red"))    

marked_but_not_in_namelist = []
for id in id_from_mark:
    if id not in id_from_namelist:   
        marked_but_not_in_namelist.append(id)

print(colored("Marked ID but not in class - {}!".format(marked_but_not_in_namelist), "red"))

In class but not marked - []!
Marked ID but not in class - []!


### Remove version history
Before you backup.

In [ ]:
## remove fill start with control- or mark- and end with .json in base_path_questions recursively.
import os
for path, currentDirectory, files in os.walk(base_path_questions):
    for file in files:
        if file.startswith("control-") or file.startswith("mark-"):
            os.remove(os.path.join(path, file))

### Reset everything (Danger)
Remove mark.js and control.js

In [ ]:
# import os
# for path, currentDirectory, files in os.walk(base_path_questions):
    
#     for file in files:       
#         if file == "control.json" or file == "mark.json":
#             os.remove(os.path.join(path, file))